# Drzewa zachowań: aplikacje ROS

Ten notebook pokazuje wzorzec aplikacji opartej na drzewie zachowań: robot zbiera dane, reaguje na priorytety, uruchamia akcje ROS i zmienia kontekst pracy systemu. Nadal korzystamy z zaadaptowanej paczki `ros_fun_py_trees_ros_tutorials`.

## Jak rozpoznawać ćwiczenia

Ćwiczenia w tym notebooku są oznaczone nagłówkiem `Ćwiczenie` i poziomą linią nad sekcją. Każde z nich ma trzy części: **Cel**, **Do zrobienia** oraz **Sprawdź**. Najpierw uruchom przykład bazowy, potem wykonaj opisaną zmianę i zweryfikuj efekt w kolejnych komórkach.

In [ ]:
# Uruchom tę komórkę na początku notebooka.
# Dodaje źródła warsztatu do PYTHONPATH i w razie potrzeby przebudowuje paczkę ROS.
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROS_DISTRO = os.environ.get("ROS_DISTRO", "jazzy")
WORKSPACE = Path("/home/ubuntu/turtlebot3_ws")
SOURCE_DIR = WORKSPACE / "src" / "jupyter_notebooks"

if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))


def bash(command, check=True):
    full = f"source /opt/ros/{ROS_DISTRO}/setup.bash; " \
           f"source {WORKSPACE}/install/setup.bash 2>/dev/null || true; " \
           f"{command}"
    return subprocess.run(["bash", "-lc", full], text=True, check=check)


def ensure_ros_fun_built():
    if shutil.which("ros2") is None:
        print("Nie widzę komendy ros2. Ten notebook trzeba uruchomić w kontenerze ROS.")
        return
    marker = WORKSPACE / "install" / "ros_fun" / "lib" / "ros_fun" / "tree-data-gathering"
    if marker.exists():
        print("ros_fun ma już zainstalowane tutoriale py_trees.")
        return
    print("Buduję ros_fun, aby ros2 launch widział nowe skrypty tutoriali...")
    bash(f"cd {WORKSPACE} && colcon build --symlink-install --packages-select ros_fun")
    print("Gotowe. Terminale uruchamiane z tego notebooka będą źródłować install/setup.bash.")

ensure_ros_fun_built()


In [ ]:
from run_in_term import run_lxterminal


def ros(command, check=True):
    return bash(command, check=check)


def terminal(command):
    escaped = command.replace("'", "'\\''")
    run_lxterminal(
        f"bash -lc 'source /opt/ros/{ROS_DISTRO}/setup.bash; "
        f"source {WORKSPACE}/install/setup.bash; {escaped}'"
    )


def stop_py_trees_tutorials():
    pattern = "[r]os2 launch ros_fun|[p]y-trees-tree-watcher|[p]y-trees-blackboard-watcher|[t]ree-data-gathering|[t]ree-battery-check|[t]ree-action-clients|[t]ree-context-switching|[t]ree-docking-cancelling-failing|[t]ree-dynamic-application-loading|[m]ock-battery|[m]ock-dashboard|[m]ock-led-strip|[m]ock-docking-controller|[m]ock-move-base|[m]ock-rotation-controller|[m]ock-safety-sensors"
    ros(f"pkill -TERM -f '{pattern}' || true", check=False)
    ros("sleep 1", check=False)
    ros(f"pkill -KILL -f '{pattern}' || true", check=False)


## Akcja jako zachowanie w drzewie

Tutorial 5 dodaje zadanie `Scan`. Zdarzenie przychodzi przez temat `/dashboard/scan`, a drzewo uruchamia akcję `/rotate` i równolegle publikuje sygnał dla paska LED.

## Dashboard mock robota

Po uruchomieniu launch file'a tutoriala w środowisku VNC powinno pojawić się okno `Dashboard`. Przyciski `Scan` i `Cancel` publikują odpowiednio na `/dashboard/scan` i `/dashboard/cancel`, a suwak baterii zmienia parametr mocka baterii. Komórki z `ros2 topic pub` robią to samo z poziomu notebooka, więc możesz używać GUI albo komend.


<img src="./images/py_trees_ros_tutorials/tutorial-five-action-clients.png" width="70%">


In [ ]:
import py_trees
from ros_fun_py_trees_ros_tutorials.five_action_clients import tutorial_create_root as create_action_tree

root = create_action_tree()
print(py_trees.display.unicode_tree(root))


In [ ]:
stop_py_trees_tutorials()
terminal("ros2 launch ros_fun tutorial_five_action_clients_launch.py")


In [ ]:
# Alternatywnie do kliknięcia Scan w dashboardzie: wyślij żądanie skanowania z komórki.
ros("ros2 topic pub --once /dashboard/scan std_msgs/msg/Empty '{}'", check=False)


In [ ]:
# Podejrzyj akcję i efekt powiadomienia.
ros("ros2 action list | grep rotate", check=False)
ros("timeout 5 ros2 topic echo --once /led_strip/display", check=False)


---

## Ćwiczenie 1: zmień zakończenie skanowania

**Cel:** zobaczyć, jak drobna zmiana w gałęzi drzewa wpływa na widoczne zachowanie robota.

**Do zrobienia:** w pliku `ros_fun_py_trees_ros_tutorials/five_action_clients.py` znajdź gałąź `scan_celebrate`. Obecna wersja czeka 3 sekundy i miga na zielono. Skróć pauzę do 1 sekundy i zmień kolor końcowy na `purple`.

**Sprawdź:** przebuduj paczkę, uruchom tutorial 5 i ponownie wyślij zdarzenie na `/dashboard/scan`.

## Introspekcja drzewa

`py_trees_ros` udostępnia narzędzia do obserwowania drzewa i blackboardu. W ROS Jazzy `py-trees-tree-watcher` potrzebuje aktywnego strumienia snapshotów z węzła `/tree`, dlatego najpierw włączamy odpowiednie parametry, a dopiero potem uruchamiamy watcher w osobnym terminalu.

In [ ]:
ros("ros2 param set /tree default_snapshot_stream True", check=False)
ros("ros2 param set /tree default_snapshot_blackboard_data True", check=False)
ros("ros2 param set /tree default_snapshot_blackboard_activity True", check=False)
terminal("py-trees-tree-watcher -a -s -b /tree/snapshots")
# Alternatywnie:
# terminal("py-trees-blackboard-watcher --visited")


## Przełączanie kontekstu

Tutorial 6 dodaje zachowanie `ScanContext`. Przy wejściu do gałęzi skanowania ustawia parametr `/safety_sensors/enabled`, a przy wyjściu przywraca jego poprzednią wartość.

<img src="./images/py_trees_ros_tutorials/tutorial-six-context-switching.png" width="70%">


In [ ]:
from ros_fun_py_trees_ros_tutorials.six_context_switching import tutorial_create_root as create_context_tree

root = create_context_tree()
print(py_trees.display.unicode_tree(root))


In [ ]:
stop_py_trees_tutorials()
terminal("ros2 launch ros_fun tutorial_six_context_switching_launch.py")


In [ ]:
ros("ros2 param get /safety_sensors enabled", check=False)
ros("ros2 topic pub --once /dashboard/scan std_msgs/msg/Empty '{}'", check=False)
ros("sleep 1; ros2 param get /safety_sensors enabled", check=False)


---

## Ćwiczenie 2: popraw czytelność gałęzi skanowania

**Cel:** sprawdzić, jak nazwy zachowań i dodatkowe sygnały pomagają diagnozować działanie drzewa.

**Do zrobienia:** w pliku `ros_fun_py_trees_ros_tutorials/six_context_switching.py` znajdź gałąź:

```python
scanning.add_children([scan_context_switch, scan_rotate, flash_blue])
```

Dodaj do niej drugi sygnał LED w innym kolorze albo zmień nazwę `Context Switch` na opis, który lepiej tłumaczy zmianę parametru.

**Sprawdź:** uruchom watcher drzewa i potwierdź, że zmiana jest widoczna w strukturze drzewa lub w zachowaniu LED.

## Anulowanie zadania

Tutorial 7 rozbudowuje aplikację o zdarzenie `/dashboard/cancel`. To dobry moment na dyskusję: anulowanie nie oznacza wyłącznie przerwania akcji, bo robot musi jeszcze wrócić do bezpiecznego stanu.

In [ ]:
from ros_fun_py_trees_ros_tutorials.seven_docking_cancelling_failing import tutorial_create_root as create_cancel_tree

root = create_cancel_tree()
print(py_trees.display.unicode_tree(root))


In [ ]:
stop_py_trees_tutorials()
terminal("ros2 launch ros_fun tutorial_seven_docking_cancelling_failing_launch.py")


In [ ]:
ros("ros2 topic pub --once /dashboard/scan std_msgs/msg/Empty '{}'", check=False)
ros("sleep 2; ros2 topic pub --once /dashboard/cancel std_msgs/msg/Empty '{}'", check=False)


---

## Ćwiczenie 3: zaprojektuj szybsze anulowanie

**Cel:** przeanalizować, gdzie w drzewie powinien być obsługiwany priorytetowy sygnał anulowania.

**Do zrobienia:** w obecnej wersji anulowanie działa tylko w wybranej części przebiegu zadania. Zaprojektuj zmianę: gdzie trzeba przenieść albo skopiować gałąź sprawdzającą `cancel2bb`, żeby aplikacja szybciej reagowała na `/dashboard/cancel`?

**Sprawdź:** narysuj zmienione drzewo albo wprowadź zmianę w kodzie i obejrzyj efekt w `py-trees-tree-watcher`.

In [ ]:
stop_py_trees_tutorials()
